# Theme: From Understanding Humans → To Guiding Them

In [57]:
# Import Libraries
import pandas as pd
import numpy as np
from scipy.sparse import hstack

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report


## Train

In [41]:
# Load the Train Data
df = pd.read_csv('/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv')

In [42]:
# Perform EDA, Remove Null Data as there are small portion of it, or cuz of Text
# Reset the Index
df = df.dropna().drop('id', axis=1).reset_index(drop=True)

In [43]:
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [44]:
wordnet =WordNetLemmatizer()
stopword = set(stopwords.words('english'))

In [45]:
import re

In [46]:
# Function for Cleaning Text Data
def preprocess_text(text):
  text = re.sub(r'[^a-zA-Z\s]',' ',text)
  text = text.lower()
  tokens = word_tokenize(text)
  tokens = [wordnet.lemmatize(word)for word in tokens if not word in stopword]
  return ' '.join(tokens)

In [47]:
# Store Cleaned Text in seprate Column
df['clean_text'] = df['journal_text'].apply(preprocess_text)

In [48]:
# WOrd Embedding -- Word ->Vector
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_text = vectorizer.fit_transform(df['clean_text'])

In [49]:
# Encode Categorical Column
df = pd.get_dummies(df, columns=[
    'ambience_type',
    'previous_day_mood',
    'face_emotion_hint',
    'reflection_quality',
    'time_of_day'
])

In [50]:
# Standardisation
scaler =StandardScaler()
num_cols = ['duration_min','sleep_hours','energy_level','stress_level']
df[num_cols] = scaler.fit_transform(df[num_cols])

In [51]:
# Final Matrix for training

X_others = df.drop(columns=['journal_text','clean_text','emotional_state','intensity'])
X = hstack((X_text, X_others.astype(float))) # Indenpendent
# Dependent
y_state = df['emotional_state']
y_intensity = df['intensity']

In [52]:
# Dimentionality Reduction
svd = TruncatedSVD(n_components=100,n_iter=5, random_state=42)
X_reduced = svd.fit_transform(X)

In [53]:
# Train - Train Testt Split
X_train, X_val, y_train, y_val = train_test_split(X_reduced, y_state, test_size=0.2, random_state=42)
_, _, y_train_intensity, y_val_intensity = train_test_split(X_reduced, y_intensity, test_size=0.2, random_state=42)

In [54]:
# Emotional State Model
model_state = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
model_state.fit(X_train, y_train)

RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=42)

In [55]:
# Evaluate the Model
y_pred_state = model_state.predict(X_val)

accuracy_score = accuracy_score(y_val, y_pred_state)
print(accuracy_score)

0.5518867924528302


In [58]:
# INTENSITY MODEL
model_intensity = RandomForestRegressor(n_estimators=100, random_state=42)
model_intensity.fit(X_train, y_train_intensity)


# EVALUATE REGRESSION

y_pred_intensity = model_intensity.predict(X_val)

mse = mean_squared_error(y_val_intensity, y_pred_intensity)
print("MSE:", mse)

MSE: 1.9543169811320755


# Test

In [59]:
df_test = pd.read_csv('/content/arvyax_test_inputs_120.xlsx - Sheet1.csv')
df_test.head(2)

,id,journal_text,ambience_type,duration_min,sleep_hours,energy_level,stress_level,time_of_day,previous_day_mood,face_emotion_hint,reflection_quality
0,10001,woke up feeling more organized mentally. i was...,cafe,4,8.5,3,1,night,mixed,happy_face,vague
1,10002,started off distracted most of the time. this ...,mountain,4,8.5,1,2,afternoon,mixed,happy_face,clear


In [60]:
df_test = df_test.drop('id', axis=1).reset_index(drop=True)

In [61]:
df_test['clean_text'] = df_test['journal_text'].apply(preprocess_text)
X_test_text = vectorizer.transform(df_test['clean_text'])

In [62]:
df_test = pd.get_dummies(df_test, columns=[
    'ambience_type',
    'previous_day_mood',
    'face_emotion_hint',
    'reflection_quality',
    'time_of_day'
])
num_cols = ['duration_min','sleep_hours','energy_level','stress_level']
df_test[num_cols] = scaler.transform(df_test[num_cols])
X_others_test = df_test.drop(columns=['journal_text','clean_text'])

In [63]:
# align the column
X_others_test = X_others_test.reindex(columns=X_others.columns, fill_value=0)

In [64]:
X_test = hstack((X_test_text, X_others_test.astype(float))) # Actual Test Data

In [65]:
X_test_reduced= svd.transform(X_test)

In [66]:
# Actual PREDICTIONS on given Dataset

y_test_state_pred = model_state.predict(X_test_reduced)
y_test_intensity_pred = model_intensity.predict(X_test_reduced)

In [67]:
#  CONFIDENCE SCORE

probs = model_state.predict_proba(X_test_reduced)
confidence = np.max(probs, axis=1)

In [68]:
# Uncertinity flag
uncertain_flag = (confidence < 0.6).astype(int)

In [69]:
# Decision Layer
def decision_layer(state, intensity):
    intensity = round(intensity)


    action = "reflect"
    timing = "later_today"

    if state in ["stress", "anxiety"]:
        if intensity >= 4:
            action = "breathing exercise"
            timing = "now"
        elif intensity >= 2:
            action = "short walk"
            timing = "within_15_min"
        else:
            action = "journaling"
            timing = "later_today"

    elif state in ["sad", "sadness"]:
        if intensity >= 4:
            action = "talk to someone"
            timing = "tonight"
        else:
            action = "light journaling"
            timing = "later_today"

    elif state in ["happy", "calm"]:
        action = "continue routine"
        timing = "now"

    elif state in ["tired", "low_energy"]:
        action = "rest"
        timing = "now"

    elif state == "neutral":
        if intensity >= 3:
            action = "deep work"
            timing = "within_15_min"
        else:
            action = "plan tasks"
            timing = "later_today"

    return action, timing

In [70]:
# Support Message
def generate_message(state, intensity, action, timing):
    intensity = round(intensity)

    return f"You seem {state} with intensity {intensity}. " \
           f"It might help to try {action} {timing.replace('_', ' ')}."

In [71]:
results = []

for i in range(X_test_reduced.shape[0]):

    state = y_test_state_pred[i]
    intensity = float(y_test_intensity_pred[i])
    conf = float(confidence[i])
    uncertain = int(uncertain_flag[i])

    # Decision layer
    action, timing = decision_layer(state, intensity)

    # Support message
    message = generate_message(state, intensity, action, timing)

    results.append({
        "predicted_state": state,
        "predicted_intensity": round(intensity, 2),
        "action": action,
        "timing": timing,
        "confidence": round(conf, 3),
        "uncertain_flag": uncertain,
        "message": message
    })

final_df = pd.DataFrame(results)
print(final_df.head())

  predicted_state  ...                                            message
0         focused  ...  You seem focused with intensity 4. It might he...
1         neutral  ...  You seem neutral with intensity 3. It might he...
2         focused  ...  You seem focused with intensity 3. It might he...
3         neutral  ...  You seem neutral with intensity 3. It might he...
4         neutral  ...  You seem neutral with intensity 3. It might he...

[5 rows x 7 columns]


In [72]:
# FINAL CSV EXPORT
final_df.to_csv("predictions.csv", index=False)